# Advanced EDA — VLC Simulation Dataset

Dataset: `part-00000.parquet`

## 1. Setup

In [ ]:
!pip -q install pyarrow plotly missingno scikit-learn --upgrade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import missingno as msno
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.manifold import TSNE
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 200)

## 2. Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
path = list(uploaded.keys())[0] if 'uploaded' in dir() else "part-00000.parquet"
df = pd.read_parquet(path)
df.shape

## 3. Structural Overview

In [ ]:
df.head()

In [ ]:
df.info(memory_usage="deep")

In [ ]:
prefixes = sorted(set(c.split("_")[0] for c in df.columns))
groups = {p: [c for c in df.columns if c.startswith(p + "_")] for p in prefixes}
for p, cols in groups.items():
    print(p, len(cols))

In [ ]:
df.describe(include="all").T

## 4. Missing Values

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["missing"] > 0].sort_values("pct", ascending=False)
missing_df

In [ ]:
msno.matrix(df)
plt.show()

In [ ]:
msno.heatmap(df)
plt.show()

## 5. Duplicates & Constant Columns

In [ ]:
df.duplicated().sum()

In [ ]:
nunique = df.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
constant_cols

In [ ]:
id_like_cols = [c for c in df.columns if df[c].nunique() == len(df)]
id_like_cols

## 6. Categorical Columns

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string", "bool"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in constant_cols + id_like_cols]
cat_cols

In [ ]:
for c in cat_cols:
    print(c)
    print(df[c].value_counts(dropna=False))
    print()

In [ ]:
n = len(cat_cols)
fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(18, 4 * ((n + 2) // 3)))
axes = axes.flatten()
for i, c in enumerate(cat_cols):
    order = df[c].astype(str).value_counts().index
    sns.countplot(data=df, x=c, order=order, ax=axes[i])
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].set_title(c)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

## 7. Numerical Columns — Distributions

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in constant_cols + id_like_cols]
len(num_cols)

In [ ]:
skew_kurt = pd.DataFrame({
    "skew": df[num_cols].skew(),
    "kurtosis": df[num_cols].kurtosis()
}).sort_values("skew", ascending=False)
skew_kurt

In [ ]:
sample_cols = num_cols[:20]
fig, axes = plt.subplots((len(sample_cols) + 3) // 4, 4, figsize=(20, 4 * ((len(sample_cols) + 3) // 4)))
axes = axes.flatten()
for i, c in enumerate(sample_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(c)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

In [ ]:
sample_cols2 = num_cols[20:40]
fig, axes = plt.subplots((len(sample_cols2) + 3) // 4, 4, figsize=(20, 4 * ((len(sample_cols2) + 3) // 4)))
axes = axes.flatten()
for i, c in enumerate(sample_cols2):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(c)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

## 8. Outlier Detection (IQR & Boxplots)

In [ ]:
def iqr_outlier_pct(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).mean() * 100

outlier_pct = pd.Series({c: iqr_outlier_pct(df[c].dropna()) for c in num_cols}).sort_values(ascending=False)
outlier_pct.head(20)

In [ ]:
top_outlier_cols = outlier_pct.head(12).index.tolist()
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for i, c in enumerate(top_outlier_cols):
    sns.boxplot(y=df[c], ax=axes[i])
    axes[i].set_title(c)
plt.tight_layout()
plt.show()

## 9. Correlation Analysis

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(24, 20))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2)
plt.show()

In [ ]:
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
top_corr = corr_pairs.abs().sort_values(ascending=False).head(30)
top_corr_full = corr_pairs.loc[top_corr.index]
top_corr_full

## 10. Group-wise Analysis

In [ ]:
group_col = "meta_scenario_type"
key_metrics = ["comm_sum_rate_mbps", "phys_avg_snr_db", "loc_error_3d_m", "label_joint_sum_rate_mbps", "opt_total_power_w"]
key_metrics = [c for c in key_metrics if c in df.columns]
df.groupby(group_col)[key_metrics].agg(["mean", "median", "std"])

In [ ]:
fig, axes = plt.subplots(1, len(key_metrics), figsize=(6 * len(key_metrics), 5))
for ax, c in zip(axes, key_metrics):
    sns.boxplot(data=df, x=group_col, y=c, ax=ax)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
pipeline_col = "meta_pipeline_mode"
df.groupby(pipeline_col)[key_metrics].mean()

In [ ]:
fig = px.sunburst(df, path=[group_col, pipeline_col], values="meta_sample_index")
fig.show()

## 11. Feasibility / Status Flags

In [ ]:
flag_cols = ["opt_converged", "opt_feasible", "label_qos_feasible", "label_joint_converged", "label_joint_feasible", "meta_is_valid"]
flag_cols = [c for c in flag_cols if c in df.columns]
df[flag_cols].mean()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, c in enumerate(flag_cols):
    df[c].value_counts().plot.pie(autopct="%1.1f%%", ax=axes[i])
    axes[i].set_title(c)
    axes[i].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["loc_status"], df["alloc_qos_status"])

## 12. Relationship Between Key Continuous Variables

In [ ]:
pair_cols = ["phys_avg_snr_db", "comm_sum_rate_mbps", "loc_error_3d_m", "opt_total_power_w", "label_joint_sum_rate_mbps"]
pair_cols = [c for c in pair_cols if c in df.columns]
sns.pairplot(df[pair_cols + [group_col]].dropna(), hue=group_col, corner=True)
plt.show()

In [ ]:
fig = px.scatter_3d(df.dropna(subset=["loc_est_x", "loc_est_y", "loc_est_z"]), x="loc_est_x", y="loc_est_y", z="loc_est_z", color=group_col, opacity=0.6)
fig.show()

## 13. Dimensionality Reduction

In [ ]:
X = df[num_cols].fillna(df[num_cols].median())
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)
pca.explained_variance_ratio_

In [ ]:
plot_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
plot_df[group_col] = df[group_col].values
fig = px.scatter(plot_df, x="PC1", y="PC2", color=group_col, opacity=0.7)
fig.show()

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=num_cols, columns=["PC1", "PC2"])
loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(15)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca")
tsne_res = tsne.fit_transform(X_scaled)
tsne_df = pd.DataFrame(tsne_res, columns=["TSNE1", "TSNE2"])
tsne_df[group_col] = df[group_col].values
fig = px.scatter(tsne_df, x="TSNE1", y="TSNE2", color=group_col, opacity=0.7)
fig.show()

## 14. Multivariate Outlier Detection

In [ ]:
iso = IsolationForest(contamination=0.03, random_state=42)
outlier_pred = iso.fit_predict(X_scaled)
df["is_outlier"] = outlier_pred == -1
df["is_outlier"].value_counts()

In [ ]:
plot_df["is_outlier"] = df["is_outlier"].values
fig = px.scatter(plot_df, x="PC1", y="PC2", color="is_outlier", opacity=0.7)
fig.show()

## 15. Feature Relationships With a Target Metric

In [ ]:
target = "label_joint_sum_rate_mbps"
target_corr = df[num_cols].corr()[target].drop(target).sort_values(key=abs, ascending=False)
target_corr.head(20)

In [ ]:
top_feats = target_corr.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, c in enumerate(top_feats):
    sns.regplot(data=df, x=c, y=target, scatter_kws={"alpha": 0.4}, ax=axes[i])
    axes[i].set_title(f"{c} vs {target}")
plt.tight_layout()
plt.show()

## 16. Time-based View

In [ ]:
df["meta_timestamp"] = pd.to_datetime(df["meta_timestamp"])
df.set_index("meta_timestamp")[key_metrics].resample("1min").mean().plot(subplots=True, figsize=(14, 12))
plt.tight_layout()
plt.show()

## 17. Summary Report

In [ ]:
summary = {
    "n_rows": len(df),
    "n_cols": df.shape[1],
    "n_numeric": len(num_cols),
    "n_categorical": len(cat_cols),
    "n_constant_cols": len(constant_cols),
    "n_duplicate_rows": int(df.duplicated().sum()),
    "pct_multivariate_outliers": float(df["is_outlier"].mean() * 100),
    "n_scenario_types": df[group_col].nunique(),
}
pd.Series(summary)